# 09-10_weather_api_ingestion_and_storage

Fetch weather forecasts from OpenWeather, normalize the results, and store/query them in MySQL.

In [ ]:
import os
import pandas as pd
import requests
from sqlalchemy import create_engine, text

# Set these in your environment before running:
# OPENWEATHER_API_KEY, DB_USER, DB_PASSWORD, DB_HOST

In [ ]:

API_key = os.getenv("OPENWEATHER_API_KEY", "ADD_YOUR_KEY_HERE")
lat = 52.52
lon = 13.405

url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"

response = requests.get(url)
response.status_code

In [ ]:
response.json()

In [ ]:
# show keys for population data
response.json().keys()

In [ ]:
response.json()['list']

In [ ]:
response.json()['list'][0].keys()

In [ ]:
forecast = response.json()['list'][0]

forecast['dt_txt']
forecast['main']['temp']
forecast['weather'][0]['description']
forecast['pop']

forecast_data = []
for entry in response.json()['list']:
    forecast_data.append({
        'datetime': entry['dt_txt'],
        'temp': entry['main']['temp'],
        'description': entry['weather'][0]['description'],
        'pop': entry['pop']
    })

In [ ]:
forecast_df = pd.DataFrame(forecast_data)
forecast_df.head()

In [ ]:
import pandas as pd
import requests

API_KEY = os.getenv("OPENWEATHER_API_KEY", "ADD_YOUR_KEY_HERE")   # replace this
# rotate the old one from your notebook if it was exposed

def get_weather_for_multiple_cities(cities_df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """
    Takes a DataFrame with at least:
        city, latitude, longitude
    and returns a forecast DataFrame for multiple cities.
    """
    
    all_forecasts = []

    for _, row in cities_df.iterrows():
        city = row["city"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = (
            f"https://api.openweathermap.org/data/2.5/forecast"
            f"?lat={lat}&lon={lon}&appid={api_key}&units=metric"
        )

        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        for entry in data["list"]:
            all_forecasts.append({
                "city": city,
                "forecast_time": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_main": entry["weather"][0]["main"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", None),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"],
                "humidity": entry["main"]["humidity"],
                "visibility": entry.get("visibility", None)
            })

    return pd.DataFrame(all_forecasts)

In [ ]:
weather_df = get_weather_for_multiple_cities(df_cities, API_KEY)
weather_df.head()

In [ ]:
def get_weather_for_multiple_cities(cities_df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    all_forecasts = []

    for _, row in cities_df.iterrows():
        city = row["city"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = (
            f"https://api.openweathermap.org/data/2.5/forecast"
            f"?lat={lat}&lon={lon}&appid={api_key}&units=metric"
        )

        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        for entry in data["list"]:
            temp = entry["main"]["temp"]
            pop = entry.get("pop", None)
            rain_3h = entry.get("rain", {}).get("3h", 0)
            wind = entry["wind"]["speed"]

            all_forecasts.append({
                "city": city,
                "forecast_time": entry["dt_txt"],
                "temperature_c": temp,
                "feels_like_c": entry["main"]["feels_like"],
                "weather_description": entry["weather"][0]["description"],
                "pop": pop,
                "rain_3h_mm": rain_3h,
                "wind_speed": wind,
                "humidity": entry["main"]["humidity"],
                "visibility": entry.get("visibility", None),
                "is_cold": temp < 10,
                "is_rainy": (pop is not None and pop >= 0.5) or rain_3h > 0,
                "is_windy": wind > 8
            })

    return pd.DataFrame(all_forecasts)

In [ ]:
weather_df.groupby("city")[["temperature_c", "pop", "rain_3h_mm", "wind_speed"]].mean()

In [ ]:
def get_weather_for_cities(df_cities, API_key):
    """
    Takes a DataFrame with columns:
    city, latitude, longitude

    Returns a DataFrame with 5-day / 3-hour forecast
    for all cities in df_cities.
    """
    
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city = row["city"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city": city,
                "datetime": entry["dt_txt"],
                "temp": entry["main"]["temp"],
                "feels_like": entry["main"]["feels_like"],
                "description": entry["weather"][0]["description"],
                "weather_main": entry["weather"][0]["main"],
                "pop": entry.get("pop", 0),
                "rain_3h": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"],
                "humidity": entry["main"]["humidity"],
                "visibility": entry.get("visibility", None)
            })

    return pd.DataFrame(all_forecast_data)

In [ ]:
weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.tail()

In [ ]:
weather_df.shape

In [ ]:
weather_df.sort_values(["city", "datetime"]).head(20)

In [ ]:
def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city = row["city"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            temp = entry["main"]["temp"]
            pop = entry.get("pop", 0)
            rain_3h = entry.get("rain", {}).get("3h", 0)
            wind_speed = entry["wind"]["speed"]

            all_forecast_data.append({
                "city": city,
                "datetime": entry["dt_txt"],
                "temp": temp,
                "feels_like": entry["main"]["feels_like"],
                "description": entry["weather"][0]["description"],
                "pop": pop,
                "rain_3h": rain_3h,
                "wind_speed": wind_speed,
                "humidity": entry["main"]["humidity"],
                "is_cold": temp < 10,
                "is_rainy": (pop >= 0.5) or (rain_3h > 0),
                "is_windy": wind_speed > 8
            })

    return pd.DataFrame(all_forecast_data)

In [ ]:
from sqlalchemy import create_engine, text
engine_no_db = create_engine("mysql+pymysql://${DB_USER}:${DB_PASSWORD}@${DB_HOST}/")


In [ ]:
from sqlalchemy import create_engine, text

# connect to the database
engine = create_engine("mysql+pymysql://${DB_USER}:${DB_PASSWORD}@${DB_HOST}/cities_db")

In [ ]:
query = """
SELECT
    c.city_id,
    c.city,
    cc.latitude,
    cc.longitude
FROM cities c
JOIN city_coordinates cc
    ON c.city_id = cc.city_id
"""
df_cities = pd.read_sql(query, engine)
df_cities.head()

In [ ]:
weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.to_sql("weather", con=engine, if_exists="append", index=False)

In [ ]:
def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city_id = row["city_id"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city_id": city_id,
                "forecast_datetime": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_main": entry["weather"][0]["main"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", 0),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"],
                "humidity": entry["main"]["humidity"],
                "visibility": entry.get("visibility", None)
            })

    weather_df = pd.DataFrame(all_forecast_data)
    weather_df["forecast_datetime"] = pd.to_datetime(weather_df["forecast_datetime"])
    weather_df["scraped_at"] = pd.Timestamp.now()
    weather_df = weather_df.drop_duplicates(subset=["city_id", "forecast_datetime"])

    return weather_df

In [ ]:
weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.to_sql("weather", con=engine, if_exists="replace", index=False)

In [ ]:
pd.read_sql("SHOW COLUMNS FROM weather;", engine)

In [ ]:
query = """
SELECT
    c.city,
    w.forecast_datetime,
    w.temperature_c,
    w.weather_description
FROM weather w
JOIN cities c
    ON w.city_id = c.city_id
"""
pd.read_sql(query, engine).head()

In [ ]:
# check unique city_ids in weather table
pd.read_sql("SELECT DISTINCT city_id FROM weather;", engine)


In [ ]:
# number of columns in weather table
pd.read_sql("SHOW COLUMNS FROM weather;", engine)

In [ ]:
def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city_id = row["city_id"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city_id": city_id,
                "forecast_datetime": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", 0),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"],
                "scraped_at": pd.Timestamp.now()
            })

    weather_df = pd.DataFrame(all_forecast_data)
    weather_df["forecast_datetime"] = pd.to_datetime(weather_df["forecast_datetime"])
    weather_df = weather_df.drop_duplicates(subset=["city_id", "forecast_datetime"])

    return weather_df

In [ ]:
weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.to_sql("weather", con=engine, if_exists="replace", index=False)

In [ ]:
cities = {
    "Berlin": "https://en.wikipedia.org/wiki/Berlin",
    "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
    "Munich": "https://en.wikipedia.org/wiki/Munich"
}

df_population = scrape_cities_with_population(cities)

cities_df = df_population[["city", "country"]].copy()
cities_df["city_id"] = range(1, len(cities_df) + 1)
cities_df = cities_df[["city_id", "city", "country"]]

city_coordinates_df = df_population[["city", "latitude", "longitude"]].copy()
city_coordinates_df = city_coordinates_df.merge(
    cities_df[["city_id", "city"]],
    on="city",
    how="left"
)
city_coordinates_df = city_coordinates_df[["city_id", "latitude", "longitude"]]

city_population_df = df_population[["city", "population", "scraped_at"]].copy()
city_population_df = city_population_df.merge(
    cities_df[["city_id", "city"]],
    on="city",
    how="left"
)
city_population_df = city_population_df[["city_id", "population", "scraped_at"]]

cities_df.to_sql("cities", con=engine, if_exists="replace", index=False)
city_coordinates_df.to_sql("city_coordinates", con=engine, if_exists="replace", index=False)
city_population_df.to_sql("city_population", con=engine, if_exists="replace", index=False)

query = """
SELECT
    c.city_id,
    c.city,
    cc.latitude,
    cc.longitude
FROM cities c
JOIN city_coordinates cc
    ON c.city_id = cc.city_id
"""
df_cities = pd.read_sql(query, engine)

def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city_id = row["city_id"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city_id": city_id,
                "forecast_datetime": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", 0),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"],
                "scraped_at": pd.Timestamp.now()
            })

    weather_df = pd.DataFrame(all_forecast_data)
    weather_df["forecast_datetime"] = pd.to_datetime(weather_df["forecast_datetime"])
    weather_df = weather_df.drop_duplicates(subset=["city_id", "forecast_datetime"])

    return weather_df

weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.to_sql("weather", con=engine, if_exists="replace", index=False)
weather_df.head()

In [ ]:
cities = {
    "Berlin": "https://en.wikipedia.org/wiki/Berlin",
    "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
    "Munich": "https://en.wikipedia.org/wiki/Munich"
}

df_population = scrape_cities_with_population(cities)

cities_df = df_population[["city", "country"]].copy()
cities_df["city_id"] = range(1, len(cities_df) + 1)
cities_df = cities_df[["city_id", "city", "country"]]

city_coordinates_df = df_population[["city", "latitude", "longitude"]].copy()
city_coordinates_df = city_coordinates_df.merge(
    cities_df[["city_id", "city"]],
    on="city",
    how="left"
)
city_coordinates_df = city_coordinates_df[["city_id", "latitude", "longitude"]]

city_population_df = df_population[["city", "population"]].copy()
city_population_df = city_population_df.merge(
    cities_df[["city_id", "city"]],
    on="city",
    how="left"
)
city_population_df = city_population_df[["city_id", "population"]]

cities_df.to_sql("cities", con=engine, if_exists="replace", index=False)
city_coordinates_df.to_sql("city_coordinates", con=engine, if_exists="replace", index=False)
city_population_df.to_sql("city_population", con=engine, if_exists="replace", index=False)

query = """
SELECT
    c.city_id,
    c.city,
    cc.latitude,
    cc.longitude
FROM cities c
JOIN city_coordinates cc
    ON c.city_id = cc.city_id
"""
df_cities = pd.read_sql(query, engine)

def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city_id = row["city_id"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city_id": city_id,
                "forecast_datetime": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", 0),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"]
            })

    weather_df = pd.DataFrame(all_forecast_data)
    weather_df["forecast_datetime"] = pd.to_datetime(weather_df["forecast_datetime"])
    weather_df = weather_df.drop_duplicates(subset=["city_id", "forecast_datetime"])

    return weather_df

weather_df = get_weather_for_cities(df_cities, API_key)
weather_df.to_sql("weather", con=engine, if_exists="replace", index=False)
weather_df.head()

In [ ]:
def get_weather_for_cities(df_cities, API_key):
    all_forecast_data = []

    for _, row in df_cities.iterrows():
        city_id = row["city_id"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        response = requests.get(url)
        data = response.json()

        for entry in data["list"]:
            all_forecast_data.append({
                "city_id": city_id,
                "forecast_datetime": entry["dt_txt"],
                "temperature_c": entry["main"]["temp"],
                "feels_like_c": entry["main"]["feels_like"],
                "weather_description": entry["weather"][0]["description"],
                "pop": entry.get("pop", 0),
                "rain_3h_mm": entry.get("rain", {}).get("3h", 0),
                "wind_speed": entry["wind"]["speed"]
            })

    city_weather_df = pd.DataFrame(all_forecast_data)
    city_weather_df["forecast_datetime"] = pd.to_datetime(city_weather_df["forecast_datetime"])
    city_weather_df = city_weather_df.drop_duplicates(subset=["city_id", "forecast_datetime"])

    return city_weather_df

In [ ]:
city_weather_df = get_weather_for_cities(df_cities, API_key)
city_weather_df.to_sql("city_weather", con=engine, if_exists="replace", index=False)
city_weather_df.head()